In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [4]:
pd.set_option('display.max_columns', 60)

In [37]:
df = pd.read_csv("D:/CODING/Virtual_py/PROJECTS/real_estate_ml/backend/kolkata/kolkata_cleaned_v5.csv")

In [11]:
new_df = df[['BEDROOM_NUM', 'AGE', 'TOTAL_FLOOR',
       'TOTAL_LANDMARK_COUNT',
       'BALCONY_NUM', 'FLOOR_NUM', 'PRICE_Cr', 'AREA_SQFT',
       'LATITUDE', 'LONGITUDE', 'OWNTYPE_label',
       'FACING_LABEL', 'FURNISH_LABEL',  'PROPERTY_type', 'CITY_LABELS', 
       'CITY_NUM', 'LOCALITY']]

In [12]:
new_df.head()

,BEDROOM_NUM,AGE,TOTAL_FLOOR,TOTAL_LANDMARK_COUNT,BALCONY_NUM,FLOOR_NUM,PRICE_Cr,AREA_SQFT,LATITUDE,LONGITUDE,OWNTYPE_label,FACING_LABEL,FURNISH_LABEL,PROPERTY_type,CITY_LABELS,CITY_NUM,LOCALITY
0,3.0,6,4.0,29.0,2.0,2.0,0.50,1287.0,22.629509,88.479623,Cooperative Society,East,Fully furnished,Residential Apartment,Kolkata East,28,Rajarhat
1,3.0,2,15.0,8.0,1.0,7.0,0.45,1377.0,88.210954,22.493402,Cooperative Society,East,Other,Residential Apartment,Kolkata South,27,Maheshtala
2,2.0,3,3.0,48.0,4.0,3.0,0.35,810.0,22.471111,88.372903,Cooperative Society,South-East,Other,Residential Apartment,Kolkata South,27,Baishnabghata Bye Lane
3,3.0,5,3.0,50.0,1.0,1.0,0.75,1208.0,22.496720,88.370570,Cooperative Society,East,Fully furnished,Residential Apartment,Kolkata South,27,Raja SC Mallick ROAD
4,2.0,3,5.0,50.0,0.0,0.0,0.50,827.0,22.505744,88.356716,Cooperative Society,North-West,Unfurnished,Residential Apartment,Kolkata South,27,Lake Gardens


In [13]:
numerical_cols = ["BEDROOM_NUM", "AGE", "TOTAL_FLOOR", "TOTAL_LANDMARK_COUNT", "BALCONY_NUM", "FLOOR_NUM", "PRICE_Cr","AREA_SQFT", "CITY_NUM"]
categorical_cols_ore = ["OWNTYPE_label", "FURNISH_LABEL"]
categorical_cols_ohe = ["FACING_LABEL", "PROPERTY_type", "CITY_LABELS", "LOCALITY"]

In [38]:
df.head()

,PREFERENCE,DESCRIPTION,TRANSACT_TYPE,BEDROOM_NUM,AGE,TOTAL_FLOOR,PROP_NAME,PROP_HEADING,SECONDARY_TAGS,TOTAL_LANDMARK_COUNT,BALCONY_NUM,FLOOR_NUM,PRICE_Cr,AREA_SQFT,PRICE_PER_SQFT(INR),OWNTYPE_label,LATITUDE,LONGITUDE,Cooperative Society,Power of Attorney,Other,Leasehold,FACING_LABEL,FACING_label_North,FACING_label_North-East,FACING_label_North-West,FACING_label_Other,FACING_label_South,FACING_label_South-East,FACING_label_South-West,FACING_label_West,FURNISH_LABEL,FURNISH_label_Other,FURNISH_label_Semi-furnished,FURNISH_label_Unfurnished,PROPERTY_type,PROPERTY_TYPE_Independent/Builder Floor,PROPERTY_TYPE_Residential Apartment,PROPERTY_TYPE_Residential Land,CITY_LABELS,CITY_Kolkata East,CITY_Kolkata North,CITY_Kolkata South,CITY_Kolkata West,LANDMARK_DETAILS,CITY_NUM,LOCALITY
0,S,3bhk flat on the second floor of a four-Storey...,1.0,3.0,6,4.0,team taurus kabya,3 BHK Flat in Rajarhat,"['READY TO MOVE', 'RESALE', 'RERA | HIRA']",29.0,2.0,2.0,0.50,1287.0,3885.00,Cooperative Society,22.629509,88.479623,1,0,0,0,East,0,0,0,0,0,0,0,0,Fully furnished,0,0,0,Residential Apartment,0,1,0,Kolkata East,1,0,0,0,"['ReligiousPlace', 'Hospital', 'OfficeComplex'...",28,Rajarhat
1,S,"3bhk, 2 bath, semi-Furnished, 7th floor (Of 15...",1.0,3.0,2,15.0,eden city maheshtala,3 BHK Flat in Maheshtala,"['READY TO MOVE', 'RESALE', 'RERA | HIRA']",8.0,1.0,7.0,0.45,1377.0,3267.97,Cooperative Society,88.210954,22.493402,1,0,0,0,East,0,0,0,0,0,0,0,0,Other,1,0,0,Residential Apartment,0,1,0,Kolkata South,0,0,1,0,"['MetroStation', 'Shopping', 'Connectivity', '...",27,Maheshtala
2,S,3bhk flat/apartment for sale in baisnabghata b...,1.0,2.0,3,3.0,baisnabghata,2 BHK Flat in Baishnabghata Bye Lane,"['READY TO MOVE', 'RESALE']",48.0,4.0,3.0,0.35,810.0,4320.99,Cooperative Society,22.471111,88.372903,1,0,0,0,South-East,0,0,0,0,0,1,0,0,Other,1,0,0,Residential Apartment,0,1,0,Kolkata South,0,0,1,0,"['MetroStation', 'ReligiousPlace', 'ATM', 'Hos...",27,Baishnabghata Bye Lane
3,S,This is your chance to own a 3 bhk residential...,1.0,3.0,5,3.0,moana by varada group,3 BHK Flat in Raja SC Mallick ROAD,"['UNDER CONSTRUCTION', 'RESALE']",50.0,1.0,1.0,0.75,1208.0,6208.61,Cooperative Society,22.496720,88.370570,1,0,0,0,East,0,0,0,0,0,0,0,0,Fully furnished,0,0,0,Residential Apartment,0,1,0,Kolkata South,0,0,1,0,"['ReligiousPlace', 'Hospital', 'Attraction']",27,Raja SC Mallick ROAD
4,S,2bhk flat/apartment for sale in lake gardens c...,1.0,2.0,3,5.0,lake gardens chs,2 BHK Flat in Lake Gardens,"['READY TO MOVE', 'RESALE']",50.0,0.0,0.0,0.50,827.0,6045.95,Cooperative Society,22.505744,88.356716,1,0,0,0,North-West,0,0,1,0,0,0,0,0,Unfurnished,0,0,1,Residential Apartment,0,1,0,Kolkata South,0,0,1,0,"['MetroStation', 'ReligiousPlace', 'ATM', 'Hos...",27,Lake Gardens


In [40]:
df = df.drop(columns = ["PREFERENCE",	"DESCRIPTION",	"TRANSACT_TYPE", "PROP_NAME",	"PROP_HEADING",	"SECONDARY_TAGS", "OWNTYPE_label"	,"LATITUDE", "PRICE_PER_SQFT(INR)",	"LONGITUDE", "FACING_LABEL", "FURNISH_LABEL", "PROPERTY_type", "CITY_LABELS", "LANDMARK_DETAILS"])

In [41]:
df.head()

,BEDROOM_NUM,AGE,TOTAL_FLOOR,TOTAL_LANDMARK_COUNT,BALCONY_NUM,FLOOR_NUM,PRICE_Cr,AREA_SQFT,Cooperative Society,Power of Attorney,Other,Leasehold,FACING_label_North,FACING_label_North-East,FACING_label_North-West,FACING_label_Other,FACING_label_South,FACING_label_South-East,FACING_label_South-West,FACING_label_West,FURNISH_label_Other,FURNISH_label_Semi-furnished,FURNISH_label_Unfurnished,PROPERTY_TYPE_Independent/Builder Floor,PROPERTY_TYPE_Residential Apartment,PROPERTY_TYPE_Residential Land,CITY_Kolkata East,CITY_Kolkata North,CITY_Kolkata South,CITY_Kolkata West,CITY_NUM,LOCALITY
0,3.0,6,4.0,29.0,2.0,2.0,0.50,1287.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,28,Rajarhat
1,3.0,2,15.0,8.0,1.0,7.0,0.45,1377.0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,27,Maheshtala
2,2.0,3,3.0,48.0,4.0,3.0,0.35,810.0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,27,Baishnabghata Bye Lane
3,3.0,5,3.0,50.0,1.0,1.0,0.75,1208.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,27,Raja SC Mallick ROAD
4,2.0,3,5.0,50.0,0.0,0.0,0.50,827.0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,27,Lake Gardens


In [42]:
new_df = pd.get_dummies(df, columns = ["LOCALITY"], drop_first=True)

In [43]:
new_df.shape

(4515, 422)

In [44]:
X = new_df.drop(columns = ["PRICE_Cr"])
y = new_df["PRICE_Cr"]

In [45]:
y_log = np.log1p(y)

In [46]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [47]:
X_scaled = pd.DataFrame(X_scaled, columns = X.columns)


In [48]:
X_scaled.head()

,BEDROOM_NUM,AGE,TOTAL_FLOOR,TOTAL_LANDMARK_COUNT,BALCONY_NUM,FLOOR_NUM,AREA_SQFT,Cooperative Society,Power of Attorney,Other,Leasehold,FACING_label_North,FACING_label_North-East,FACING_label_North-West,FACING_label_Other,FACING_label_South,FACING_label_South-East,FACING_label_South-West,FACING_label_West,FURNISH_label_Other,FURNISH_label_Semi-furnished,FURNISH_label_Unfurnished,PROPERTY_TYPE_Independent/Builder Floor,PROPERTY_TYPE_Residential Apartment,PROPERTY_TYPE_Residential Land,CITY_Kolkata East,CITY_Kolkata North,CITY_Kolkata South,CITY_Kolkata West,CITY_NUM,...,LOCALITY_Tram Depot,LOCALITY_Ultadanga,LOCALITY_Uniworld City,LOCALITY_Usha Park,LOCALITY_Uttarpara,LOCALITY_Uttarpara Kotrung,LOCALITY_VIP Haldiram,LOCALITY_VIP Haldirams,LOCALITY_VIP Road,LOCALITY_Vivekananda Nagar,LOCALITY_Vivekananda Park,LOCALITY_Zarda Bagan,LOCALITY_baguihati jyangra,LOCALITY_bd block,LOCALITY_be... block,LOCALITY_ca block,LOCALITY_ca.. block,LOCALITY_deshpriya park,LOCALITY_hindustha road,LOCALITY_joka metro,LOCALITY_kalighat,LOCALITY_kalyani a block,LOCALITY_mayfair road,LOCALITY_near rabindra sarovar metro Tollygung road,LOCALITY_near southern Avenue,LOCALITY_near southern Avenue and saratbose road crossing,LOCALITY_on embypass,LOCALITY_purbachal,LOCALITY_rajathat chowmatha,LOCALITY_telipukur
0,0.535007,1.713407,-0.591469,-0.038645,1.342612,-0.585702,0.098600,0.182804,-0.147392,-0.042131,-0.0969,-0.454504,-0.177563,-0.613274,-0.325692,-0.331896,-0.353686,-0.241082,-0.261991,-0.353686,-0.224697,-0.315187,-0.136848,0.299211,-0.230617,1.487023,-0.534408,-0.880603,-0.160945,1.049714,...,-0.014884,-0.036478,-0.021051,-0.014884,-0.033296,-0.014884,-0.029778,-0.021051,-0.025786,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.021051,-0.014884
1,0.535007,-0.268412,0.827629,-1.367666,-0.412162,0.484570,0.263422,0.182804,-0.147392,-0.042131,-0.0969,-0.454504,-0.177563,-0.613274,-0.325692,-0.331896,-0.353686,-0.241082,-0.261991,2.827370,-0.224697,-0.315187,-0.136848,0.299211,-0.230617,-0.672485,-0.534408,1.135585,-0.160945,-0.189684,...,-0.014884,-0.036478,-0.021051,-0.014884,-0.033296,-0.014884,-0.029778,-0.021051,-0.025786,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.021051,-0.014884
2,-0.822173,0.227042,-0.720478,1.163803,4.852159,-0.371648,-0.774955,0.182804,-0.147392,-0.042131,-0.0969,-0.454504,-0.177563,-0.613274,-0.325692,-0.331896,2.827370,-0.241082,-0.261991,2.827370,-0.224697,-0.315187,-0.136848,0.299211,-0.230617,-0.672485,-0.534408,1.135585,-0.160945,-0.189684,...,-0.014884,-0.036478,-0.021051,-0.014884,-0.033296,-0.014884,-0.029778,-0.021051,-0.025786,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.021051,-0.014884
3,0.535007,1.217952,-0.720478,1.290377,-0.412162,-0.799756,-0.046077,0.182804,-0.147392,-0.042131,-0.0969,-0.454504,-0.177563,-0.613274,-0.325692,-0.331896,-0.353686,-0.241082,-0.261991,-0.353686,-0.224697,-0.315187,-0.136848,0.299211,-0.230617,-0.672485,-0.534408,1.135585,-0.160945,-0.189684,...,-0.014884,-0.036478,-0.021051,-0.014884,-0.033296,-0.014884,-0.029778,-0.021051,-0.025786,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.014884,-0.029778,-0.021051,-0.014884
4,-0.822173,0.227042,-0.462461,1.290377,-2.166936,-1.013811,-0.743822,0.182804,-0.147392,-0.042131,-0.0969,-0.454504,-0.177563,1.630593,-0.325692,-0.331896,-0.353686,-0.241082,-0.261991,-0.353686,-0.224697,3.172724,-0.136848,0.299211,-0.230617,-0.672485,-0.534408,1.135585,-0.160945,-0.189684,...,-0.014884,-0.

In [49]:
kfold = KFold(n_splits = 10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring="r2")

In [50]:
scores.mean(), scores.std()

(np.float64(0.8659774112953802), np.float64(0.019710053229994928))

In [51]:
lr = LinearRegression()

lr.fit(X_scaled, y_log)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](421,)","[ 0.03, 0.01, 0.08,..., 0.01,-0. ,-0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](421,)","['BEDROOM_NUM','AGE','TOTAL_FLOOR',...,'LOCALITY_purbachal ', 'LOCALITY_rajathat chowmatha','LOCALITY_telipukur']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.508
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,421
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,417


In [52]:
lr.coef_.shape

(421,)

In [55]:
coeff_df = pd.DataFrame(lr.coef_.reshape(1, 421), columns = X.columns).stack().reset_index().drop(columns = ["level_0"]).rename(columns = {"level_1" : "feature", 0 : "coeff"}).sort_values(by = "coeff", ascending = False)


In [ ]:
coeff_df.head()         # these are the standardize coefficients of the linear regression model.


,feature,coeff
6,AREA_SQFT,0.144768
2,TOTAL_FLOOR,0.081720
388,LOCALITY_Tollygunge,0.050983
39,LOCALITY_Alipore,0.044026
68,LOCALITY_Ballygunge,0.035926


standardize_coeff = unstandardize * (std(x) / std(y))

In [61]:
y.std()

np.float64(0.6873582714977919)

In [62]:
X["AREA_SQFT"].std()

np.float64(546.1052838681085)

In [64]:
(0.144 * 0.687 ) / 546.105

0.00018115197626830006